# NB10 — PANDA Feature Processing

Extract two-scale ConvNeXt-Tiny features for the PANDA dataset (Radboud + Karolinska). Per manuscript Section 2A, resolution-aware level selection reads `openslide.mpp-x` from each slide and chooses the pyramid level closest to each target mpp (0.5 and 2.0 μm/pixel). This ensures Karolinska slides (native 0.25 μm/pixel, 40×) and Radboud slides (native 0.5 μm/pixel, 20×) are processed at consistent physical tissue area regardless of native magnification.

Outputs `features/panda/scale0p5/{image_id}.npy` and `features/panda/scale2p0/{image_id}.npy` (768-d float16 vectors per tile) used by NB11 (MIL training).

In [ ]:
import os, sys, json, time
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tvm
from PIL import Image
import openslide

WORKSPACE  = Path(os.environ.get('WORKSPACE',  './workspace'))
PANDA_ROOT = Path(os.environ.get('PANDA_ROOT', './data/PANDA'))
OUT_DIRS = {
    'features_05': WORKSPACE / 'features' / 'panda' / 'scale0p5',
    'features_20': WORKSPACE / 'features' / 'panda' / 'scale2p0',
    'results':     WORKSPACE / 'results' / 'panda',
    'logs':        WORKSPACE / 'logs' / 'panda',
}
for d in OUT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

TILE_SIZE = 256
STRIDE    = 224
SCALES    = [0.5, 2.0]
MAX_TILES = {0.5: 1200, 2.0: 400}
BATCH_SIZE = 128
MIN_TISSUE_COVERAGE = 0.30
HSV_S_TISSUE_MIN = 20
HSV_V_WHITE_MIN  = 230
N_WORKERS = min(cpu_count() - 1, 8)
USE_AMP = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'NB10: PANDA feature processing on {DEVICE}; {cpu_count()} CPUs, using {N_WORKERS} workers')

class ConvNeXtTinyFeats(nn.Module):
    def __init__(self):
        super().__init__()
        m = tvm.convnext_tiny(weights=tvm.ConvNeXt_Tiny_Weights.DEFAULT)
        self.features = m.features
        self.gap = nn.AdaptiveAvgPool2d(1)
        for p in self.parameters(): p.requires_grad = False
        self.eval()
    @torch.no_grad()
    def forward(self, x):
        return self.gap(self.features(x)).flatten(1)

MODEL = ConvNeXtTinyFeats().to(DEVICE)
if DEVICE != 'cpu':
    MODEL = MODEL.to(memory_format=torch.channels_last)

def slide_base_mpp(slide):
    """read native microns-per-pixel from slide metadata; fall back to 0.5 if missing."""
    props = slide.properties
    for k in ('openslide.mpp-x', 'aperio.MPP'):
        v = props.get(k)
        if v is not None:
            try:
                return float(v)
            except Exception:
                pass
    return 0.5

def choose_level(slide, target_mpp, base_mpp):
    target_downsample = target_mpp / base_mpp
    return slide.get_best_level_for_downsample(target_downsample)

def hsv_tissue_mask(thumb_rgb_np):
    img = Image.fromarray(thumb_rgb_np).convert('HSV')
    a = np.array(img, dtype=np.uint8)
    S, V = a[..., 1], a[..., 2]
    return (S >= HSV_S_TISSUE_MIN) & (V < HSV_V_WHITE_MIN)

def tissue_coverage(tile_np):
    """per-tile fraction of pixels classified as tissue using HSV rule (matches NB04)."""
    img = Image.fromarray(tile_np).convert('HSV')
    a = np.array(img, dtype=np.uint8)
    S, V = a[..., 1], a[..., 2]
    mask = (S >= HSV_S_TISSUE_MIN) & (V < HSV_V_WHITE_MIN)
    return float(mask.mean())

def process_batch(tile_batch, model, device):
    tensors = []
    for tile in tile_batch:
        arr = np.array(tile).astype(np.float32) / 255.0
        t = torch.from_numpy(arr).permute(2, 0, 1)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        tensors.append((t - mean) / std)
    batch = torch.stack(tensors).to(device, non_blocking=True)
    if device != 'cpu':
        batch = batch.to(memory_format=torch.channels_last)
    with torch.no_grad():
        if USE_AMP and device != 'cpu':
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                feats = model(batch)
        else:
            feats = model(batch)
    return feats.cpu().numpy()

def already_processed(image_id):
    p05 = OUT_DIRS['features_05'] / f'{image_id}.npy'
    p20 = OUT_DIRS['features_20'] / f'{image_id}.npy'
    if p05.exists() and p20.exists():
        try:
            f05 = np.load(p05, mmap_mode='r')
            f20 = np.load(p20, mmap_mode='r')
            return f05.shape[1] == 768 and f20.shape[1] == 768
        except Exception:
            pass
    return False

def process_one_slide(image_id, image_path):
    if already_processed(image_id):
        return image_id, 'skipped', 0
    try:
        slide = openslide.OpenSlide(str(image_path))
    except Exception as e:
        return image_id, f'open_error:{e}', 0
    try:
        base_mpp = slide_base_mpp(slide)
        tiles_total = 0
        for scale in SCALES:
            scale_key = f'features_{str(scale).replace(".","").replace("p","")}'
            scale_key = 'features_05' if abs(scale - 0.5) < 1e-6 else 'features_20'
            feat_path = OUT_DIRS[scale_key] / f'{image_id}.npy'
            if feat_path.exists():
                continue
            level = choose_level(slide, scale, base_mpp)
            actual_downsample = slide.level_downsamples[level]
            level_w, level_h = slide.level_dimensions[level]
            tiles = []; tile_batch = []
            for y in range(0, level_h - TILE_SIZE + 1, STRIDE):
                for x in range(0, level_w - TILE_SIZE + 1, STRIDE):
                    if len(tiles) >= MAX_TILES[scale]:
                        break
                    x0 = int(x * actual_downsample); y0 = int(y * actual_downsample)
                    tile = slide.read_region((x0, y0), level, (TILE_SIZE, TILE_SIZE)).convert('RGB')
                    tile_np = np.array(tile)
                    if tissue_coverage(tile_np) >= MIN_TISSUE_COVERAGE:
                        tile_224 = tile.resize((224, 224), Image.BILINEAR)
                        tile_batch.append(tile_224)
                        if len(tile_batch) >= BATCH_SIZE:
                            feats = process_batch(tile_batch, MODEL, DEVICE)
                            tiles.extend(feats); tile_batch = []
                            tiles_total += len(feats)
                if len(tiles) >= MAX_TILES[scale]:
                    break
            if tile_batch:
                feats = process_batch(tile_batch, MODEL, DEVICE)
                tiles.extend(feats); tiles_total += len(feats)
            if tiles:
                np.save(feat_path, np.vstack(tiles).astype(np.float16))
            else:
                np.save(feat_path, np.zeros((0, 768), dtype=np.float16))
    except Exception as e:
        slide.close()
        return image_id, f'process_error:{e}', 0
    slide.close()
    if DEVICE != 'cpu':
        torch.cuda.empty_cache()
    return image_id, 'success', tiles_total

manifest_path = OUT_DIRS['logs'] / 'panda_manifest.csv'
if manifest_path.exists():
    df_panda = pd.read_csv(manifest_path)
else:
    train_csv = PANDA_ROOT / 'train.csv'
    if not train_csv.exists():
        print(f'[WARN] PANDA train.csv not found at {train_csv}; skipping NB10')
        raise SystemExit(0)
    df_panda = pd.read_csv(train_csv)
    df_panda['image_path'] = df_panda['image_id'].apply(lambda x: str(PANDA_ROOT / 'train_images' / f'{x}.tiff'))
    df_panda['image_exists'] = df_panda['image_path'].apply(lambda x: Path(x).exists())
    df_panda.to_csv(manifest_path, index=False)

pending = [(r['image_id'], r['image_path']) for _, r in df_panda[df_panda['image_exists']].iterrows()
           if not already_processed(r['image_id'])]
n_total = len(df_panda)
n_existing = df_panda.get('image_exists', pd.Series()).sum() if 'image_exists' in df_panda.columns else 0
n_already = sum(1 for r in df_panda.iterrows() if r[1].get('image_exists', False) and already_processed(r[1]['image_id']))
print(f'[PLAN] total: {n_total} | with images: {n_existing} | already processed: {n_already} | to process: {len(pending)}')

if not pending:
    print('[DONE] all slides already processed')
else:
    successful = 0; failed = []; t0 = time.time()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = {ex.submit(process_one_slide, image_id, image_path): image_id for image_id, image_path in pending}
        done = 0
        for fut in as_completed(futures):
            try:
                image_id, status, tiles = fut.result()
                if status == 'success':
                    successful += 1
                elif status != 'skipped':
                    failed.append((image_id, status))
            except Exception as e:
                failed.append((futures[fut], str(e)))
            done += 1
            if done % 200 == 0:
                rate = done / max(time.time() - t0, 1e-9)
                print(f'  {done}/{len(pending)} ({rate:.1f} slides/s)  successful={successful}  failed={len(failed)}')
    elapsed = time.time() - t0
    print(f'\n[DONE] extraction in {elapsed/60:.1f} min  successful={successful}  failed={len(failed)}')
    if failed:
        pd.DataFrame(failed, columns=['image_id', 'error']).to_csv(
            OUT_DIRS['logs'] / 'failed_extractions.csv', index=False)

print('NB10 complete. Next: NB11 (PANDA MIL training).')